In [1]:
from dataset import all_tables
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
full_table = all_tables('2026-01-30_2026-04-30_node_selection.csv')
full_table.to_csv('/var/www/python/Qingcheng/WFiles/Ultra/1-30_4-30_full.csv')

KeyboardInterrupt: 

In [ ]:
def train_gru_full_table( target='slack',
                          csv_path='/var/www/python/Qingcheng/WFiles/Ultra/1-30_4-30_full.csv'):

    data_df = pd.read_csv(csv_path)
    data_df['dt'] = pd.to_datetime(data_df['dt']).dt.strftime('%Y-%m-%d')

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))

    class GRUMean(nn.Module):
        def __init__(self, input_size, output_size, hidden_size=64, num_layers=2,
                     bidirectional=True, dropout=0.1):
            super(GRUMean, self).__init__()
            self.hidden_size   = hidden_size
            self.num_layers    = num_layers
            self.bidirectional = bidirectional
            self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=num_layers,
                              batch_first=True, bidirectional=bidirectional,
                              dropout=dropout if num_layers > 1 else 0.0)
            out_dim = hidden_size * (2 if bidirectional else 1)
            self.fc = nn.Linear(out_dim, output_size)

        def forward(self, x):
            x = x.unsqueeze(-1)
            out, h_n = self.gru(x)
            last = out.mean(dim=1)
            return self.fc(last)

    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)

    learning_rate  = 0.001
    num_epochs     = 50
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    hidden_size    = 16
    num_layers     = 1
    bidirectional  = True
    dropout        = 0.1
    mean_criterion = nn.MSELoss()

    valuation_models = pd.DataFrame()
    gru_val = pd.read_csv('/var/www/python/Qingcheng/WFiles/Ultra/GRU_val_2026-04-28.csv', usecols=['dt','hr','node_num'])

    gru_val['dt'] = pd.to_datetime(gru_val['dt']).dt.strftime('%Y-%m-%d')
    gru_val['node_num'] = gru_val['node_num'].astype(int)
    y_var      = [c for c in data_df.columns if 'slack' in c and c.startswith(('da_', 'rt_'))]
    index_cols = ['dtHr', 'dt', 'hr', 'node_num']
    feature_cols = [c for c in data_df.columns if c not in index_cols + y_var]
    data_df = data_df.sort_values(['dt', 'hr']).reset_index(drop=True)

    is_test = data_df[['dt', 'hr', 'node_num']].merge(
        gru_val, on=['dt', 'hr', 'node_num'], how='left', indicator=True
    )['_merge'].eq('both').values

    df_test  = data_df[is_test]
    df_train = data_df[~is_test]

    xtest,  ytest  = df_test[feature_cols],  df_test[index_cols + y_var]
    xtrain, ytrain = df_train[feature_cols], df_train[index_cols + y_var]
    xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)

    # xtrain, xtest, ytrain, ytest         = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
    # xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)

    ytest_index = ytest[index_cols].reset_index(drop=True)
    ytrain, yvalidate, ytest = ytrain[y_var], yvalidate[y_var], ytest[y_var]

    scaler_x, scaler_y = None, None
    if normalize:
        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=feature_cols)
        xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=feature_cols)
        xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=feature_cols)
        ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=y_var)
        yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=y_var)
        ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=y_var)

    X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
    Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
    X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
    Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
    X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
    Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

    g = torch.Generator()
    g.manual_seed(42)
    train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
    validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
    test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
    input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]

    mean_model     = GRUMean(input_size, output_size, hidden_size=hidden_size, num_layers=num_layers,
                             bidirectional=bidirectional, dropout=dropout).to(device)
    mean_optimizer = optim.Adam(mean_model.parameters(), lr=learning_rate)

    best_val_loss, patience, no_improve = float("inf"), 5, 0
    for epoch in range(num_epochs):
        mean_model.train()
        for inputs, labels in train_loader:
            mean_optimizer.zero_grad()
            loss = mean_criterion(mean_model(inputs), labels)
            loss.backward()
            mean_optimizer.step()
        mean_model.eval()
        val_loss = sum(mean_criterion(mean_model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
        if val_loss < best_val_loss:
            best_val_loss, no_improve = val_loss, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    mean_preds, _ = predict_and_collect(mean_model, test_loader, mean_criterion, scaler_y)

    if normalize and scaler_y is not None:
        ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=y_var)

    mean_df = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
    result  = pd.concat([ytest_index, ytest.reset_index(drop=True), mean_df], axis=1)
    result["dtHr"] = pd.to_datetime(result["dtHr"])
    result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
    result["hr"]   = result["dtHr"].dt.hour + 1
    valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    direction_tag = "bi" if bidirectional else "uni"
    valuation_models["model"] = f"gru_{direction_tag}_{hidden_size}h_{num_layers}L_{target}"

    return valuation_models


In [10]:
final_model = train_gru_full_table()

KeyboardInterrupt: 

In [ ]:
import subprocess
import pandas as pd
import tempfile, os

# Load both files
full_df = pd.read_csv('/var/www/python/Qingcheng/WFiles/Ultra/1-30_4-30_full.csv')
val_df  = pd.read_csv('/var/www/python/Qingcheng/WFiles/Ultra/GRU_val_2026-04-28.csv')

# Write filtered CSVs to temp files and upload
uploads = [
    (full_df, 'gs://ve_fourier/temp/full_table_1-30_4-30.csv'),
    (val_df,  'gs://ve_fourier/temp/GRU_val_2026-04-28.csv'),
]
for df, dst in uploads:
    with tempfile.NamedTemporaryFile(suffix='.csv', delete=False) as f:
        df.to_csv(f.name, index=False)
        res = subprocess.run(['gsutil', 'cp', f.name, dst], capture_output=True, text=True)
        print(dst, '→', 'OK' if res.returncode == 0 else res.stderr.strip())
        os.unlink(f.name)

gs://ve_fourier/temp/full_table_1-30_4-30.csv → OK
gs://ve_fourier/temp/GRU_val_2026-04-28.csv → OK


In [ ]:
import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
sys.path.insert(0, '/var/www/python/Qingcheng/QCTest/Ultra')
from custom import run_in_kubernetes, get_metrics
from datetime import datetime

def train_gru_full_table_k8s(node_num, dt):
    import pandas as pd
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split

    csv_path = 'gs://ve_fourier/temp/full_table_1-30_4-30.csv'
    val_path = 'gs://ve_fourier/temp/GRU_val_2026-04-28.csv'

    data_df = pd.read_csv(csv_path)
    data_df['dt'] = pd.to_datetime(data_df['dt']).dt.strftime('%Y-%m-%d')

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))

    class GRUMean(nn.Module):
        def __init__(self, input_size, output_size, hidden_size=64, num_layers=2,
                     bidirectional=True, dropout=0.1):
            super(GRUMean, self).__init__()
            self.hidden_size   = hidden_size
            self.num_layers    = num_layers
            self.bidirectional = bidirectional
            self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=num_layers,
                              batch_first=True, bidirectional=bidirectional,
                              dropout=dropout if num_layers > 1 else 0.0)
            out_dim = hidden_size * (2 if bidirectional else 1)
            self.fc = nn.Linear(out_dim, output_size)

        def forward(self, x):
            x = x.unsqueeze(-1)
            out, _ = self.gru(x)
            last = out.mean(dim=1)
            return self.fc(last)

    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)

    learning_rate  = 0.001
    num_epochs     = 50
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    hidden_size    = 16
    num_layers     = 1
    bidirectional  = True
    dropout        = 0.1
    mean_criterion = nn.MSELoss()

    gru_val = pd.read_csv(val_path, usecols=['dt', 'hr', 'node_num'])
    gru_val['dt']       = pd.to_datetime(gru_val['dt']).dt.strftime('%Y-%m-%d')
    gru_val['node_num'] = gru_val['node_num'].astype(int)

    y_var        = [c for c in data_df.columns if 'slack' in c and c.startswith(('da_', 'rt_'))]
    index_cols   = ['dtHr', 'dt', 'hr', 'node_num']
    feature_cols = [c for c in data_df.columns if c not in index_cols + y_var]
    data_df      = data_df.sort_values(['dt', 'hr']).reset_index(drop=True)

    is_test = data_df[['dt', 'hr', 'node_num']].merge(
        gru_val, on=['dt', 'hr', 'node_num'], how='left', indicator=True
    )['_merge'].eq('both').values

    df_test  = data_df[is_test]
    df_train = data_df[~is_test]

    xtest,  ytest  = df_test[feature_cols],  df_test[index_cols + y_var]
    xtrain, ytrain = df_train[feature_cols], df_train[index_cols + y_var]
    xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)

    ytest_index = ytest[index_cols].reset_index(drop=True)
    ytrain, yvalidate, ytest = ytrain[y_var], yvalidate[y_var], ytest[y_var]

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=feature_cols)
    xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=feature_cols)
    xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=feature_cols)
    ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=y_var)
    yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=y_var)
    ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=y_var)

    X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
    Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
    X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
    Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
    X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
    Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

    g = torch.Generator()
    g.manual_seed(42)
    train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
    validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
    test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)

    input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
    mean_model     = GRUMean(input_size, output_size, hidden_size=hidden_size, num_layers=num_layers,
                             bidirectional=bidirectional, dropout=dropout).to(device)
    mean_optimizer = optim.Adam(mean_model.parameters(), lr=learning_rate)

    best_val_loss, patience, no_improve = float("inf"), 5, 0
    for epoch in range(num_epochs):
        mean_model.train()
        for inputs, labels in train_loader:
            mean_optimizer.zero_grad()
            loss = mean_criterion(mean_model(inputs), labels)
            loss.backward()
            mean_optimizer.step()
        mean_model.eval()
        val_loss = sum(mean_criterion(mean_model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
        if val_loss < best_val_loss:
            best_val_loss, no_improve = val_loss, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    mean_preds, _ = predict_and_collect(mean_model, test_loader, mean_criterion, scaler_y)
    ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=y_var)

    mean_df = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
    result  = pd.concat([ytest_index, ytest.reset_index(drop=True), mean_df], axis=1)
    result["dtHr"] = pd.to_datetime(result["dtHr"])
    result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
    result["hr"]   = result["dtHr"].dt.hour + 1
    result         = result.groupby(["dt", "hr", "node_num"]).max().reset_index()
    result["model"] = "gru_bi_16h_1L_full_table"
    return result

# Single job — the function trains on all nodes and returns all test predictions
result_full,val = run_in_kubernetes([('2026-04-30', 463)], train_gru_full_table_k8s)
result_full.to_csv(f'/var/www/python/Qingcheng/WFiles/Ultra/GRU_val_full_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
print(get_metrics(result_full))

Here the job starts!
2026-05-05 15:39:18.065078
  record_id 50623
We gonna upload data into cloud sql!
data upload done!
docker: 'docker rmi' requires at least 1 argument

Usage:  docker rmi [OPTIONS] IMAGE [IMAGE...]

See 'docker rmi --help' for more information

Login Succeeded
 
WARNING! Your credentials are stored unencrypted in '/home/qingcheng/.docker/config.json'.
Configure a credential helper to remove this warning. See
https://docs.docker.com/go/credential-store/


Using default tag: latest
latest: Pulling from movetocloud-999/fourier/ve_2024_nn
Digest: sha256:22c06995af4e5d82cc39deb8db214c0f8193dd36eef898519bacea228b671c5a
Status: Image is up to date for us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
 
 #0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 382B done
#1 WARN: ConsistentInstructionCasing: C

In [ ]:
def train_transformer_full_table_k8s(node_num, dt):
    import math
    import pandas as pd
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split

    csv_path = 'gs://ve_fourier/temp/full_table_1-30_4-30.csv'
    val_path = 'gs://ve_fourier/temp/GRU_val_2026-04-28.csv'

    data_df = pd.read_csv(csv_path)
    data_df['dt'] = pd.to_datetime(data_df['dt']).dt.strftime('%Y-%m-%d')

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))

    # ── model classes ───────────────────────────────────────────

    class PositionalEncoding(nn.Module):
        """Standard sinusoidal positional encoding."""
        def __init__(self, d_model, max_len=5000):
            super(PositionalEncoding, self).__init__()
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                                 (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer("pe", pe.unsqueeze(0))

        def forward(self, x):
            return x + self.pe[:, :x.size(1), :]


    class TransformerMean(nn.Module):
        """
        Transformer encoder for mean prediction.
        Treats the flat feature vector as a length-input_size sequence with 1 feature per step,
        embeds each step into d_model, applies positional encoding, runs a stack of
        TransformerEncoder layers, then projects a pooled [CLS] representation to outputs.
        """
        def __init__(self, input_size, output_size, d_model=64, nhead=4,
                     num_layers=2, dim_feedforward=128, dropout=0.1):
            super(TransformerMean, self).__init__()
            self.d_model     = d_model
            self.input_proj  = nn.Linear(1, d_model)
            self.pos_encoder = PositionalEncoding(d_model, max_len=max(input_size + 8, 64))
            self.cls_token   = nn.Parameter(torch.zeros(1, 1, d_model))
            nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True,
                activation="gelu",
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.norm        = nn.LayerNorm(d_model)
            self.fc          = nn.Linear(d_model, output_size)

        def forward(self, x):
            # x shape: (B, input_size)
            B = x.size(0)
            x = x.unsqueeze(-1)                         # (B, input_size, 1)
            x = self.input_proj(x)                      # (B, input_size, d_model)
            cls = self.cls_token.expand(B, 1, -1)       # (B, 1, d_model)
            x = torch.cat([cls, x], dim=1)              # (B, input_size+1, d_model)
            x = self.pos_encoder(x)
            x = self.transformer(x)
            cls_rep = self.norm(x[:, 0, :])             # (B, d_model)
            return self.fc(cls_rep)                     # (B, output_size)


    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)

    # ── hyper-parameters ────────────────────────────────────────
    learning_rate   = 0.001
    num_epochs      = 50
    normalize       = True
    device          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Transformer hyper-parameters
    d_model         = 64
    nhead           = 4
    num_layers      = 2
    dim_feedforward = 128
    dropout         = 0.1

    mean_criterion  = nn.MSELoss()

    # ── load validation index ───────────────────────────────────
    gru_val = pd.read_csv(val_path, usecols=['dt', 'hr', 'node_num'])
    gru_val['dt']       = pd.to_datetime(gru_val['dt']).dt.strftime('%Y-%m-%d')
    gru_val['node_num'] = gru_val['node_num'].astype(int)

    y_var        = [c for c in data_df.columns if 'slack' in c and c.startswith(('da_', 'rt_'))]
    index_cols   = ['dtHr', 'dt', 'hr', 'node_num']
    feature_cols = [c for c in data_df.columns if c not in index_cols + y_var]
    data_df      = data_df.sort_values(['dt', 'hr']).reset_index(drop=True)

    is_test = data_df[['dt', 'hr', 'node_num']].merge(
        gru_val, on=['dt', 'hr', 'node_num'], how='left', indicator=True
    )['_merge'].eq('both').values

    df_test  = data_df[is_test]
    df_train = data_df[~is_test]

    xtest,  ytest  = df_test[feature_cols],  df_test[index_cols + y_var]
    xtrain, ytrain = df_train[feature_cols], df_train[index_cols + y_var]
    xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)

    ytest_index = ytest[index_cols].reset_index(drop=True)
    ytrain, yvalidate, ytest = ytrain[y_var], yvalidate[y_var], ytest[y_var]

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=feature_cols)
    xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=feature_cols)
    xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=feature_cols)
    ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=y_var)
    yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=y_var)
    ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=y_var)

    X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
    Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
    X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
    Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
    X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
    Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

    g = torch.Generator()
    g.manual_seed(42)
    train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
    validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
    test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)

    input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]
    mean_model = TransformerMean(
        input_size, output_size,
        d_model=d_model, nhead=nhead,
        num_layers=num_layers, dim_feedforward=dim_feedforward,
        dropout=dropout,
    ).to(device)
    mean_optimizer = optim.Adam(mean_model.parameters(), lr=learning_rate)

    # ── train with early stopping ───────────────────────────────
    best_val_loss, patience, no_improve = float("inf"), 5, 0
    for epoch in range(num_epochs):
        mean_model.train()
        for inputs, labels in train_loader:
            mean_optimizer.zero_grad()
            loss = mean_criterion(mean_model(inputs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mean_model.parameters(), 1.0)
            mean_optimizer.step()
        mean_model.eval()
        with torch.no_grad():
            val_loss = sum(mean_criterion(mean_model(inp), lbl).item()
                           for inp, lbl in validate_loader) / len(validate_loader)
        if val_loss < best_val_loss:
            best_val_loss, no_improve = val_loss, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    mean_preds, _ = predict_and_collect(mean_model, test_loader, mean_criterion, scaler_y)
    ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=y_var)

    mean_df = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])
    result  = pd.concat([ytest_index, ytest.reset_index(drop=True), mean_df], axis=1)
    result["dtHr"] = pd.to_datetime(result["dtHr"])
    result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
    result["hr"]   = result["dtHr"].dt.hour + 1
    result         = result.groupby(["dt", "hr", "node_num"]).max().reset_index()
    result["model"] = f"transformer_d{d_model}_h{nhead}_L{num_layers}_full_table"
    return result


# Single job — the function trains on all nodes and returns all test predictions
result_full, val = run_in_kubernetes([('2026-04-30', 463)], train_transformer_full_table_k8s)
result_full.to_csv(f'/var/www/python/Qingcheng/WFiles/Ultra/Transformer_val_full_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
print(get_metrics(result_full))